# Sequence Creation and Update Test

This notebook creates sample candle data, builds a `Sequence`, registers indicators, and emulates sequence updates.

In [7]:
# Imports
from tradingbot.core.sequence import Interval, SequenceBuilder
from tradingbot.core.indicators import build_popular_indicators
from tradingbot.core.ticker import Ticker

In [8]:
# Generate 75 sample candles with realistic OHLCV data
import random
import math
import time

random.seed(42)

sample_candles = []
timestamp = 1711700000
open_price = 100.0

for i in range(75):
    # Simulate realistic price movement
    daily_return = random.gauss(0, 1.5)  # Mean 0, std 1.5
    open_price = open_price + daily_return

    high = open_price + abs(random.gauss(0, 1.2))
    low = open_price - abs(random.gauss(0, 1.2))
    close = open_price + random.gauss(0, 0.8)
    volume = random.uniform(1000, 3000)

    sample_candles.append(
        {
            "timestamp": timestamp,
            "open": round(open_price, 2),
            "high": round(max(open_price, high, close), 2),
            "low": round(min(open_price, low, close), 2),
            "close": round(close, 2),
            "volume": round(volume, 2),
        }
    )
    timestamp += 60  # 60-second intervals

builder = SequenceBuilder()
indicators = build_popular_indicators()


def assert_close(label, actual, expected, rel_tol=1e-6):
    assert actual is not None, f"FAIL  {label}: got None, expected {expected:.6f}"
    if abs(expected) > 0:
        assert math.isclose(actual, expected, rel_tol=rel_tol), (
            f"FAIL  {label}: expected {expected:.6f}, got {actual:.6f} "
            f"(delta={abs(actual - expected):.2e})"
        )
    print(f"PASS  {label}")


def latest_indicator_value(points):
    if not points:
        return None
    return points[-1].value


def check_indicator_point_lengths(ticker, stage):
    all_points = ticker.get_all_indicator_values()
    lengths = {name: len(points) for name, points in all_points.items()}
    min_len = min(lengths.values()) if lengths else 0
    max_len = max(lengths.values()) if lengths else 0
    target = len(ticker.sequence.candles)
    print(
        f"\n[{stage}] indicator point lengths -> "
        f"min={min_len}, max={max_len}, expected={target}"
    )
    assert min_len == target and max_len == target, (
        f"FAIL  {stage}: indicator lengths must match sequence length {target}. "
        f"Found min={min_len}, max={max_len}"
    )
    print(f"PASS  {stage}: all indicator point series align with sequence length")


global_total_start = time.perf_counter()
build_start = time.perf_counter()
sequence = builder.build_sequence_from_dicts(sample_candles, Interval.MINUTE)
ticker = Ticker(
    name="notebook_sequence",
    interval=Interval.MINUTE,
    indicators=indicators,
)
build_end = time.perf_counter()

build_latency_ms = (build_end - build_start) * 1_000
print(f"Sequence built with {len(ticker.sequence.candles)} candles")
print(f"Indicators registered: {len(ticker.indicators)}")
print(f"Build latency: {build_latency_ms:.3f} ms")

print("\nIndicators in ticker:")
for ind in ticker.indicators:
    print(f"  - {ind.name}")

print("\nIndicator values (latest point):")
for name, points in ticker.get_all_indicator_values().items():
    latest = latest_indicator_value(points)
    print(f"  {name}: {latest:.6f}" if latest is not None else f"  {name}: None")

check_indicator_point_lengths(ticker, "after_build")

Sequence built with 0 candles
Indicators registered: 11
Build latency: 0.881 ms

Indicators in ticker:
  - MA20
  - MA50
  - EMA9
  - EMA21
  - VWAP
  - RSI14
  - MACD(12,26)
  - BBW(20,2.0)
  - ATR14
  - STOCH14
  - OBV

Indicator values (latest point):
  MA20: None
  MA50: None
  EMA9: None
  EMA21: None
  VWAP: None
  RSI14: None
  MACD(12,26): None
  BBW(20,2.0): None
  ATR14: None
  STOCH14: None
  OBV: None

[after_build] indicator point lengths -> min=0, max=0, expected=0
PASS  after_build: all indicator point series align with sequence length


In [10]:
# Emulate a last-candle update (triggers incremental recompute path)
ticker.poll()  # Poll to ensure indicators are computed for all candles before update
last_index = len(ticker.sequence.candles) - 1
last_candle = ticker.sequence.candles[last_index]
updated_last_candle = builder.candle_builder.build_candle(
    timestamp=last_candle.timestamp,
    open=last_candle.open,
    high=last_candle.high + 1.5,
    low=last_candle.low,
    close=last_candle.close + 2.5,  # Bump close price up
    volume=last_candle.volume + 500.0,
)

print("\n--- update_candle (incremental recompute path) ---")
print(f"Updating candle at index {last_index} (total {len(ticker.indicators)} indicators)")

start_update = time.perf_counter()
ticker.update_sequence(updated_last_candle, operation="update")
end_update = time.perf_counter()

latency_update_us = (end_update - start_update) * 1_000_000
print(f"✓ Latency: {latency_update_us:.2f} µs ({latency_update_us/1000:.3f} ms)")

print("\nSample indicator latest values after update:")
all_values = ticker.get_all_indicator_values()
for i, (name, points) in enumerate(all_values.items()):
    if i < 5:  # Show first 5
        print(points)
        latest = latest_indicator_value(points)
        print(f"  {name}: {latest:.6f}" if latest is not None else f"  {name}: None")

check_indicator_point_lengths(ticker, "after_update_candle")

ValueError: candle_api is not configured

In [ ]:
# Emulate rolling update with add_candle: drop first candle, append new one
rolling_candle = builder.candle_builder.build_candle(
    timestamp=ticker.sequence.candles[-1].timestamp + 60,
    open=ticker.sequence.candles[-1].close,
    high=ticker.sequence.candles[-1].close + abs(random.gauss(0, 1.2)),
    low=ticker.sequence.candles[-1].close - abs(random.gauss(0, 1.2)),
    close=ticker.sequence.candles[-1].close + random.gauss(0, 0.8),
    volume=random.uniform(1000, 3000),
)

print("\n--- add_candle (full recompute path) ---")
print(f"Rolling update: drop first + append new (total {len(ticker.indicators)} indicators)")

start_add = time.perf_counter()
ticker.update_sequence(rolling_candle, operation="rolling_add")
end_add = time.perf_counter()

latency_add_us = (end_add - start_add) * 1_000_000
print(f"✓ Latency: {latency_add_us:.2f} µs ({latency_add_us/1000:.3f} ms)")

print(f"\nSequence still has {len(ticker.sequence.candles)} candles")
print("Sample indicator latest values after rolling update:")
all_values = ticker.get_all_indicator_values()
for i, (name, points) in enumerate(all_values.items()):
    if i < 5:  # Show first 5
        latest = latest_indicator_value(points)
        print(f"  {name}: {latest:.6f}" if latest is not None else f"  {name}: None")

check_indicator_point_lengths(ticker, "after_add_candle")

# Summary comparison
print("\n" + "=" * 60)
print("LATENCY COMPARISON")
print("=" * 60)
print(f"update_candle (incremental):  {latency_update_us:>10.2f} µs ({latency_update_us/1000:>8.3f} ms)")
print(f"add_candle (full recompute):  {latency_add_us:>10.2f} µs ({latency_add_us/1000:>8.3f} ms)")
print(f"Ratio (add/update):           {latency_add_us/latency_update_us:>10.1f}x")
print("=" * 60)

# Whole computation timing: build + update + add
global_total_end = time.perf_counter()
whole_ms = (global_total_end - global_total_start) * 1_000
print(f"\nWhole computation time (build + update + add): {whole_ms:.3f} ms")

print("\nTest Summary:")
print(f"  • Sequence size: {len(ticker.sequence.candles)} candles")
print(f"  • Indicators: {len(ticker.indicators)}")


--- add_candle (full recompute path) ---
Rolling update: drop first + append new (total 11 indicators)
✓ Latency: 11514.42 µs (11.514 ms)

Sequence still has 75 candles
Sample indicator latest values after rolling update:
  ATR14: 2.618490
  BBW(20,2.0): 0.185241
  EMA9: 124.096320
  STOCH14: 62.060576
  VWAP: 106.072178

[after_add_candle] indicator point lengths -> min=75, max=75, expected=75
PASS  after_add_candle: all indicator point series align with sequence length

LATENCY COMPARISON
update_candle (incremental):     6074.25 µs (   6.074 ms)
add_candle (full recompute):    11514.42 µs (  11.514 ms)
Ratio (add/update):                  1.9x

Whole computation time (build + update + add): 32.613 ms

Test Summary:
  • Sequence size: 75 candles
  • Indicators: 11
